In [3]:
import keras
import tensorflow as tf
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score,precision_recall_fscore_support
from tensorflow.keras.layers import TextVectorization, Embedding

In [4]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [5]:
train.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


In [6]:
test.head()

,id,keyword,location,text
0,0,NaN,NaN,Just happened a terrible car crash
1,2,NaN,NaN,"Heard about #earthquake is different cities, s..."
2,3,NaN,NaN,"there is a forest fire at spot pond, geese are..."
3,9,NaN,NaN,Apocalypse lighting. #Spokane #wildfires
4,11,NaN,NaN,Typhoon Soudelor kills 28 in China and Taiwan


* Here we cannot use texting data because the text data not have target column.
* Based on the testing data we can find the prediction but we will not have the refrnce to check how much % model predicted correctly.
* So for that we are split the train data into two parts train and test than validate the model after that then we will use the testing data again for the checkin model performance.

In [7]:
train.shape

(7613, 5)

In [8]:
test.shape

(3263, 4)

* Agend of the model is to find the sentiment of the sentnce.
* The data contains the natural distaer sentnces. so based on the sentce model have to identify the postive if not natural distaer (0) , if natural disaster negative (1)

In [9]:
train.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


In [10]:
train = train.sample(frac=1,random_state=50)

* Here we can use the frac btw 0 to 1. if we give frac=0.5 only 50% will be shuflled and other data will not be disturbed. if give 0.9 90% of data will be shuffled and if give 1 it will shuflle the all the rows.

In [11]:
train.head()

,id,keyword,location,text,target
3931,5589,flood,NaN,Internet basics: the flood defective intertiss...,1
1036,1502,body%20bags,D.C. - Baltimore - Annapolis,UPDATE: Va. firefighter on administrative leav...,0
4223,5998,hazardous,"British Columbia, Canada",Skinny Jeans are Hazardous for Your Health! #s...,1
6831,9783,trapped,Puerto Rico,Hollywood Movie About Trapped Miners Released ...,1
2032,2916,danger,"Brooklyn, NY",My take away: preservation parks r an impositi...,0


In [12]:
from sklearn.model_selection import train_test_split

In [13]:
train_text,val_text,train_label,val_label= train_test_split(train['text'].to_numpy(),
                                                            train['target'].to_numpy(),test_size=0.2,
                                                            random_state=100)

* Consider here 1.       train_text as (x_train)
  2. val_text is (x_test)
  3.train_label is (y_train)
  4.val_label is (y_test)
* And Note for doing the vectorization if our text data is in the array format the vectorization process will be easy so that's we are converting the text dat to arrry form which was in the data fram format (text data in a row).

In [14]:
train_text

array(["http://t.co/XlFi7ovhFJ VIDEO: 'We're picking up bodies from water': Rescuers are searching for hundreds\x89Û_ http://t.co/rAq4ZpdvKe",
       'They absolutely demolished the sounders from start to finish',
       'As California fires rage the Forest Service sounds the alarm about rising wildfire costs http://t.co/Tft1bb4xaZ',
       ...,
       'quoted here--&gt;CNN: Purported ISIS video threatens Croatian hostage http://t.co/swVuZxi6gT',
       '@anellatulip there is a theory that makes way too much sense that says that the dwarves may be the actual origin of the blight',
       'll//ll= Pic of 16yr old PKK suicide bomber who detonated bomb in Turkey Army trench released http://t.co/uoNEbAHH3h /'],
      shape=(6090,), dtype=object)

In [15]:
train_label

array([1, 0, 1, ..., 1, 0, 1], shape=(6090,))

In [16]:
val_text

array(['#landslide while on a trip in #skardu https://t.co/nqNWkTRhsA',
       "'Without an ally near you can't use this skill.' How did you get hired? Really cause it's making every Riot staff member look incompetent.",
       "New and now: Different (FNaF fanfiction): Trixie_drowned / 2 pagesHi my awesome proxies it's ... http://t.co/366NhTg3Tz #wattpad #promo",
       ...,
       'I liked a @YouTube video from @dannyonpc http://t.co/AD38KWoGlh Battlefield Hardline - 11 NEW WEAPONS - New map - Throwingknifes!',
       "@AuntieDote @RioSlade @Locke_Wiggins @akarb74 Not if one side is set on obliteration of govt and the other on fixing it. That's too far",
       'Set our hearts ablaze and every city was a gift And every skyline was like a kiss upon the lips @\x89Û_ https://t.co/cYoMPZ1A0Z'],
      shape=(1523,), dtype=object)

In [17]:
val_label

array([0, 0, 0, ..., 0, 0, 0], shape=(1523,))

In [18]:
text_vect = TextVectorization(max_tokens=10000,
                              standardize='lower_and_strip_punctuation',
                              split='whitespace',ngrams=None,
                              output_mode='int',output_sequence_length=15)

* Here the Standardize function is internal function which will reomve the puncations from the data.
* max_tokens means we can number of words we can pass.
*spilt will say that after a space word is passed.


In [19]:
text_vect.adapt(train_text)

In [20]:
data= 'There is a flood in my area'

In [21]:
text_vect(data)

<tf.Tensor: shape=(15,), dtype=int64, numpy=
array([ 74,   9,   3, 237,   5,  13, 350,   0,   0,   0,   0,   0,   0,
         0,   0])>

* This how the vctorization happening converting text into vector.
* If give word that dont have in our trained list to the model for that word model will return output as one.

In [22]:
data= 'TajMahal'

In [23]:
text_vect(data)

<tf.Tensor: shape=(15,), dtype=int64, numpy=array([1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])>

In [24]:
Emb = Embedding(input_dim=10000,output_dim=128,input_length=15)
Emb(text_vect(data))

<tf.Tensor: shape=(15, 128), dtype=float32, numpy=
array([[-0.04288594,  0.04688642,  0.01170477, ..., -0.01400528,
        -0.00736902,  0.04575584],
       [-0.01495178, -0.00373369, -0.0119871 , ...,  0.0215492 ,
        -0.0002293 , -0.02344395],
       [-0.01495178, -0.00373369, -0.0119871 , ...,  0.0215492 ,
        -0.0002293 , -0.02344395],
       ...,
       [-0.01495178, -0.00373369, -0.0119871 , ...,  0.0215492 ,
        -0.0002293 , -0.02344395],
       [-0.01495178, -0.00373369, -0.0119871 , ...,  0.0215492 ,
        -0.0002293 , -0.02344395],
       [-0.01495178, -0.00373369, -0.0119871 , ...,  0.0215492 ,
        -0.0002293 , -0.02344395]], shape=(15, 128), dtype=float32)>

* This is the vectors for the word tajmahal.
* The parameters dimension should be same for wt we used in the text_vextorization

## RNN-Model Building

In [58]:
rnn = tf.keras.models.Sequential()
### Input layer
rnn.add(tf.keras.Input(shape=(1,),dtype=tf.string))
rnn.add(text_vect)
rnn.add(Emb)
## First Hidden layer
rnn.add(tf.keras.layers.SimpleRNN(units=64,return_sequences=True))
## Second Hidden layer
rnn.add(tf.keras.layers.LSTM(64))
## Output layer
rnn.add(tf.keras.layers.Dense(1,activation='sigmoid'))

* Here we are given in input layer shape=1 and datatype = string bcs we are actually passing the text data in the string form that data is vectorized first and passed to the embeding model the model output is now passed to the RNN model.

In [27]:
### Model Compile
rnn.compile(optimizer=tf.keras.optimizers.Adam(),
            loss= tf.keras.losses.binary_crossentropy,metrics=['accuracy'])

In [30]:
## Model Training and Validation
rnn.fit(train_text,train_label,epochs=10,batch_size=40,validation_data=(val_text,val_label))

Epoch 1/10
153/153 ━━━━━━━━━━━━━━━━━━━━ 9s 30ms/step - accuracy: 0.7489 - loss: 0.5179 - val_accuracy: 0.7754 - val_loss: 0.4732
Epoch 2/10
153/153 ━━━━━━━━━━━━━━━━━━━━ 6s 37ms/step - accuracy: 0.8813 - loss: 0.2991 - val_accuracy: 0.7754 - val_loss: 0.5516
Epoch 3/10
153/153 ━━━━━━━━━━━━━━━━━━━━ 4s 29ms/step - accuracy: 0.9376 - loss: 0.1733 - val_accuracy: 0.7617 - val_loss: 0.6002
Epoch 4/10
153/153 ━━━━━━━━━━━━━━━━━━━━ 5s 27ms/step - accuracy: 0.9609 - loss: 0.1107 - val_accuracy: 0.7413 - val_loss: 0.7010
Epoch 5/10
153/153 ━━━━━━━━━━━━━━━━━━━━ 6s 36ms/step - accuracy: 0.9709 - loss: 0.0749 - val_accuracy: 0.7544 - val_loss: 0.7743
Epoch 6/10
153/153 ━━━━━━━━━━━━━━━━━━━━ 4s 27ms/step - accuracy: 0.9724 - loss: 0.0612 - val_accuracy: 0.7492 - val_loss: 0.9165
Epoch 7/10
153/153 ━━━━━━━━━━━━━━━━━━━━ 4s 28ms/step - accuracy: 0.9773 - loss: 0.0482 - val_accuracy: 0.7518 - val_loss: 1.1902
Epoch 8/10
153/153 ━━━━━━━━━━━━━━━━━━━━ 5s 36ms/step - accuracy: 0.9749 - loss: 0.0560 - val_accu

In [32]:
np.round(rnn.predict(val_text))

48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step


array([[0.],
       [0.],
       [0.],
       ...,
       [0.],
       [0.],
       [1.]], shape=(1523, 1), dtype=float32)

* Here we are getting the values from the outlayer given sigmoid function means output values are in probability based.
* The dimension of the output is in 2D so we can't compare this output with the original data for that we need to change the dimension of the data orginal to 2D or output to 1D.
* Changing the dimension of the output data is easy for us to compare with orignal

In [34]:
val_label.ndim

1

In [41]:
pred = tf.squeeze(np.round(rnn.predict(val_text)))
pred

48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step


<tf.Tensor: shape=(1523,), dtype=float32, numpy=array([0., 0., 0., ..., 0., 0., 1.], shape=(1523,), dtype=float32)>

* Here our values are in the probability based that's we are rounding them based on if its less than 0.5 to 0 and if >0.5 to 1

In [40]:
print(pred.ndim)

1


In [42]:
accuracy_score(val_label,pred)

0.7334208798424163

In [43]:
def calculate_results(true,pred):
  accuracy = accuracy_score(true,pred)
  precision,recall,f1,_= precision_recall_fscore_support(true,pred,average='weighted')
  model_results = {'accuracy':accuracy,
                  'precision':precision,
                  'recall':recall,
                  'f1':f1}
  return model_results

* Here we are above creating function which can directly predict accuracyscore, precision, recall and f1_score and we are saving the those output in the dictonary foramt to display.

In [44]:
calculate_results(val_label,pred)

{'accuracy': 0.7334208798424163,
 'precision': 0.7354805025675013,
 'recall': 0.7334208798424163,
 'f1': 0.7341268369625915}

In [45]:
test

,id,keyword,location,text
0,0,NaN,NaN,Just happened a terrible car crash
1,2,NaN,NaN,"Heard about #earthquake is different cities, s..."
2,3,NaN,NaN,"there is a forest fire at spot pond, geese are..."
3,9,NaN,NaN,Apocalypse lighting. #Spokane #wildfires
4,11,NaN,NaN,Typhoon Soudelor kills 28 in China and Taiwan
...,...,...,...,...
3258,10861,NaN,NaN,EARTHQUAKE SAFETY LOS ANGELES ÛÒ SAFETY FASTE...
3259,10865,NaN,NaN,Storm in RI worse than last hurricane. My city...
3260,10868,NaN,NaN,Green Line derailment in Chicago http://t.co/U...
3261,10874,NaN,NaN,MEG issues Hazardous Weather Outlook (HWO) htt...


In [51]:
text_= np.array(test['text'])
text_

array(['Just happened a terrible car crash',
       'Heard about #earthquake is different cities, stay safe everyone.',
       'there is a forest fire at spot pond, geese are fleeing across the street, I cannot save them all',
       ..., 'Green Line derailment in Chicago http://t.co/UtbXLcBIuY',
       'MEG issues Hazardous Weather Outlook (HWO) http://t.co/3X6RBQJHn3',
       '#CityofCalgary has activated its Municipal Emergency Plan. #yycstorm'],
      shape=(3263,), dtype=object)

* Our data in the tabular form to predict the data we need to convert it too the array.

In [55]:
tf.squeeze(np.round(rnn.predict(text_)))

102/102 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step


<tf.Tensor: shape=(3263,), dtype=float32, numpy=array([1., 0., 1., ..., 1., 1., 1.], shape=(3263,), dtype=float32)>

* This is the output for the testing data.
* Here also we are converted the output of the 2D to 1D using the squzee function and then rounded to get the zero and one values.
* For this data set we dont have the refrence colum of target check the acccurcy so that's  we are only getting the output.

## LSTM

In [56]:
lstm = tf.keras.Sequential()
## Input layer
lstm.add(tf.keras.Input(shape=(1,),dtype=tf.string))
## Vecorized data
lstm.add(text_vect)
## Embedding data
lstm.add(Emb)
## First Hidden layer
lstm.add(tf.keras.layers.LSTM(units=64,return_sequences=True))
## Second Hidden layer
lstm.add(tf.keras.layers.LSTM(64))
## Outlayer
lstm.add(tf.keras.layers.Dense(1,activation='sigmoid'))

In [59]:
### Model Compile
lstm.compile(optimizer=tf.keras.optimizers.Adam(),
            loss= tf.keras.losses.binary_crossentropy,metrics=['accuracy'])

In [60]:
## Model Training and Validation
lstm.fit(train_text,train_label,epochs=10,batch_size=32,validation_data=(val_text,val_label))

Epoch 1/10
191/191 ━━━━━━━━━━━━━━━━━━━━ 14s 44ms/step - accuracy: 0.9103 - loss: 0.2413 - val_accuracy: 0.7498 - val_loss: 0.6655
Epoch 2/10
191/191 ━━━━━━━━━━━━━━━━━━━━ 6s 33ms/step - accuracy: 0.9677 - loss: 0.1076 - val_accuracy: 0.7511 - val_loss: 0.8974
Epoch 3/10
191/191 ━━━━━━━━━━━━━━━━━━━━ 8s 40ms/step - accuracy: 0.9722 - loss: 0.0818 - val_accuracy: 0.7406 - val_loss: 0.8752
Epoch 4/10
191/191 ━━━━━━━━━━━━━━━━━━━━ 6s 33ms/step - accuracy: 0.9768 - loss: 0.0579 - val_accuracy: 0.7301 - val_loss: 1.1729
Epoch 5/10
191/191 ━━━━━━━━━━━━━━━━━━━━ 8s 40ms/step - accuracy: 0.9777 - loss: 0.0537 - val_accuracy: 0.7538 - val_loss: 1.2334
Epoch 6/10
191/191 ━━━━━━━━━━━━━━━━━━━━ 6s 31ms/step - accuracy: 0.9806 - loss: 0.0395 - val_accuracy: 0.7341 - val_loss: 1.5297
Epoch 7/10
191/191 ━━━━━━━━━━━━━━━━━━━━ 7s 38ms/step - accuracy: 0.9816 - loss: 0.0443 - val_accuracy: 0.7485 - val_loss: 1.2161
Epoch 8/10
191/191 ━━━━━━━━━━━━━━━━━━━━ 6s 33ms/step - accuracy: 0.9811 - loss: 0.0394 - val_acc

In [63]:
pred = tf.squeeze(np.round(lstm.predict(val_text)))

48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step


In [64]:
calculate_results(val_label,pred)

{'accuracy': 0.7367038739330269,
 'precision': 0.7361882396031705,
 'recall': 0.7367038739330269,
 'f1': 0.736407307648483}

## GRU

In [66]:
gru = tf.keras.Sequential()
## Input layer
gru.add(tf.keras.Input(shape=(1,),dtype=tf.string))
## Vecorized data
gru.add(text_vect)
## Embedding data
gru.add(Emb)
## First Hidden layer
gru.add(tf.keras.layers.GRU(units=64,return_sequences=True))
## Second Hidden layer
gru.add(tf.keras.layers.GRU(64))
## Output layer
gru.add(tf.keras.layers.Dense(1,activation='sigmoid'))

In [67]:
### Model Compile
gru.compile(optimizer=tf.keras.optimizers.Adam(),
            loss = tf.keras.losses.binary_crossentropy,metrics=['accuracy'])

In [69]:
## Model Training and Validation
gru.fit(train_text,train_label,epochs=10,batch_size=32,validation_data=(val_text,val_label))

Epoch 1/10
191/191 ━━━━━━━━━━━━━━━━━━━━ 17s 53ms/step - accuracy: 0.9289 - loss: 0.1746 - val_accuracy: 0.7564 - val_loss: 0.9880
Epoch 2/10
191/191 ━━━━━━━━━━━━━━━━━━━━ 14s 72ms/step - accuracy: 0.9744 - loss: 0.0674 - val_accuracy: 0.7295 - val_loss: 0.9525
Epoch 3/10
191/191 ━━━━━━━━━━━━━━━━━━━━ 14s 71ms/step - accuracy: 0.9800 - loss: 0.0493 - val_accuracy: 0.7380 - val_loss: 1.1141
Epoch 4/10
191/191 ━━━━━━━━━━━━━━━━━━━━ 13s 33ms/step - accuracy: 0.9780 - loss: 0.0460 - val_accuracy: 0.7413 - val_loss: 1.1266
Epoch 5/10
191/191 ━━━━━━━━━━━━━━━━━━━━ 8s 41ms/step - accuracy: 0.9811 - loss: 0.0393 - val_accuracy: 0.7209 - val_loss: 1.2590
Epoch 6/10
191/191 ━━━━━━━━━━━━━━━━━━━━ 6s 33ms/step - accuracy: 0.9821 - loss: 0.0356 - val_accuracy: 0.7347 - val_loss: 1.3985
Epoch 7/10
191/191 ━━━━━━━━━━━━━━━━━━━━ 8s 41ms/step - accuracy: 0.9846 - loss: 0.0325 - val_accuracy: 0.7223 - val_loss: 1.6669
Epoch 8/10
191/191 ━━━━━━━━━━━━━━━━━━━━ 7s 38ms/step - accuracy: 0.9818 - loss: 0.0374 - val_

In [72]:
pred = tf.squeeze(np.round(gru.predict(val_text)))

48/48 ━━━━━━━━━━━━━━━━━━━━ 3s 40ms/step


In [73]:
calculate_results(val_label,pred)

{'accuracy': 0.7222586999343401,
 'precision': 0.7239878856022427,
 'recall': 0.7222586999343401,
 'f1': 0.7228962446964056}